# 01_eda — Exploration des données NHTSA Tesla FSD

## Objectif
Exploration descriptive uniquement (EDA). Pas de modélisation, pas d'inférence statistique, pas de conclusion tarifaire.

## Sources de données
Le notebook explore ce qui est présent dans `data/raw/` (côté WSL) :

- **nhtsa_sgo/level2_ADAS.csv** — rapports d'incidents ADAS (niveau rapport d'incident)
- **nhtsa_sgo/other_incidents.csv** — autres incidents (niveau rapport d'incident)
- **fars/FARS2024NationalCSV/accident.csv** — accidents mortels FARS 2024 (niveau accident)
- **fars/FARS2024NationalCSV/vehicle.csv** — véhicules impliqués FARS 2024 (niveau véhicule)
- **fars/FARS2024NationalCSV/person.csv** — personnes impliquées FARS 2024 (niveau personne)

> ⚠️ Ces données proviennent de la NHTSA (USA). Ce notebook se limite à l'EDA descriptive.
> Aucune conclusion tarifaire pour l'Europe n'est tirée ici. L'hypothèse de transférabilité USA→Europe est énoncée à titre indicatif uniquement.

In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

raw_dir = Path("data/raw")
print("Répertoire de données :", raw_dir.resolve())
print("Existe :", raw_dir.exists())
if raw_dir.exists():
    for sub in sorted(raw_dir.iterdir()):
        if sub.is_dir():
            files = [f.name for f in sub.iterdir() if f.is_file() and not f.name.endswith(":Zone.Identifier")]
            print("  ", sub.name, ":", files)

## Utilitaires de chargement

Les valeurs REDACTED / PII / Unknown sont traitées comme manquantes.
Chaque source est chargée indépendamment car leurs schémas sont incompatibles (SGO = rapport d'incident, FARS = accident/véhicule/personne).

In [ ]:
SENTINEL_VALUES = [
    "[REDACTED, MAY CONTAIN CONFIDENTIAL BUSINESS INFORMATION]",
    "[MAY CONTAIN PERSONALLY IDENTIFIABLE INFORMATION]",
    "[REDACTED]",
    "",
    " ",
    "Unknown",
    "unknown",
    "UNKNOWN",
    "Not Applicable",
    "Not applicable",
]

def _clean(df: pd.DataFrame) -> pd.DataFrame:
    return df.replace(SENTINEL_VALUES, np.nan)

def load_sgo(raw_dir: Path) -> dict:
    """Charge les fichiers SGO/ADAS (niveau rapport d'incident)."""
    out = {}
    adas = raw_dir / "nhtsa_sgo" / "level2_ADAS.csv"
    other = raw_dir / "nhtsa_sgo" / "other_incidents.csv"
    for name, f in [("level2_ADAS", adas), ("other_incidents", other)]:
        if f.exists():
            print("Chargement SGO :", f.name, "(", round(f.stat().st_size / 1e6, 1), "Mo)")
            df = pd.read_csv(f, dtype=str, low_memory=False)
            df = _clean(df)
            out[name] = df
        else:
            print("Fichier SGO introuvable :", f)
    return out

def load_fars(raw_dir: Path) -> dict:
    """Charge les fichiers FARS 2024 (accident / vehicle / person)."""
    out = {}
    fars_dir = raw_dir / "fars" / "FARS2024NationalCSV"
    if not fars_dir.exists():
        print("Répertoire FARS introuvable :", fars_dir)
        return out
    for name in ["accident", "vehicle", "person"]:
        f = fars_dir / (name + ".csv")
        if f.exists():
            print("Chargement FARS :", f.name, "(", round(f.stat().st_size / 1e6, 1), "Mo)")
            df = pd.read_csv(f, dtype=str, low_memory=False)
            df = _clean(df)
            out[name] = df
        else:
            print("Fichier FARS introuvable :", f)
    return out

sgo = load_sgo(raw_dir)
fars = load_fars(raw_dir)
print()
print("SGO chargé :", list(sgo.keys()))
print("FARS chargé :", list(fars.keys()))

## 1. EDA — Données SGO/ADAS (rapports d'incident)

Si présentes, exploration des colonnes `Automation System Engaged?`, météo, route, gravité, vitesse.

In [ ]:
SEP = "=" * 60
if sgo:
    for name, df in sgo.items():
        print()
        print(SEP)
        print("SGO —", name, ":", len(df), "lignes ×", df.shape[1], "colonnes")
        print(SEP)
        print("Colonnes :", list(df.columns)[:30], "..." if df.shape[1] > 30 else "")
        print()
        print("Valeurs manquantes (top 10) :")
        print(df.isna().sum().sort_values(ascending=False).head(10))

        if "Make" in df.columns:
            print()
            print("Top marques :")
            print(df["Make"].value_counts().head(10))

        if "Automation System Engaged?" in df.columns:
            print()
            print("Automation System Engaged? :")
            print(df["Automation System Engaged?"].value_counts(dropna=False))

        if "Highest Injury Severity Alleged" in df.columns:
            print()
            print("Gravité :")
            print(df["Highest Injury Severity Alleged"].value_counts(dropna=False))

        if "SV Precrash Speed (MPH)" in df.columns:
            sp = pd.to_numeric(df["SV Precrash Speed (MPH)"], errors="coerce")
            print()
            print("Vitesse avant crash : moyenne=", round(sp.mean(), 1), "mph, médiane=", round(sp.median(), 1), "mph, max=", round(sp.max(), 1), "mph")
else:
    print("Aucune donnée SGO chargée.")

## 2. EDA — Données FARS 2024 (accidents mortels)

Exploration des accidents, véhicules et personnes. FARS ne contient que les accidents mortels (pas d'exposition kilométrique), donc seuls des comptes bruts et des proportions sont calculés.

In [ ]:
if fars:
    acc = fars.get("accident")
    veh = fars.get("vehicle")
    per = fars.get("person")

    if acc is not None:
        print()
        print(SEP)
        print("FARS accident :", len(acc), "accidents mortels")
        print(SEP)
        print("États (top 10) :")
        print(acc["STATENAME"].value_counts().head(10))
        print()
        print("Mois (top 10) :")
        print(acc["MONTHNAME"].value_counts().head(10))
        print()
        print("Conditions météo (WEATHERNAME, top 10) :")
        print(acc["WEATHERNAME"].value_counts(dropna=False).head(10))
        print()
        print("Éclairage (LGT_CONDNAME, top 10) :")
        print(acc["LGT_CONDNAME"].value_counts(dropna=False).head(10))
        print()
        print("Total décès (FATALS) :", acc["FATALS"].astype(float).sum())

    if veh is not None:
        print()
        print(SEP)
        print("FARS vehicle :", len(veh), "véhicules impliqués")
        print(SEP)
        print("Top marques (MAKENAME) :")
        print(veh["MAKENAME"].value_counts(dropna=False).head(15))
        print()
        print("Top modèles (VPICMODELNAME) :")
        print(veh["VPICMODELNAME"].value_counts(dropna=False).head(15))
        print()
        print("Types de carrosserie (BODY_TYPNAME, top 10) :")
        print(veh["BODY_TYPNAME"].value_counts(dropna=False).head(10))

        is_tesla = veh["MAKENAME"].str.contains("Tesla", case=False, na=False)
        print()
        print("Véhicules Tesla dans FARS :", is_tesla.sum())
        if is_tesla.sum() > 0:
            print("Modèles Tesla :")
            print(veh.loc[is_tesla, "VPICMODELNAME"].value_counts(dropna=False))

    if per is not None:
        print()
        print(SEP)
        print("FARS person :", len(per), "personnes impliquées")
        print(SEP)
        print("Gravité des blessures (INJ_SEVNAME) :")
        print(per["INJ_SEVNAME"].value_counts(dropna=False))
        print()
        print("Type de personne (PER_TYPNAME) :")
        print(per["PER_TYPNAME"].value_counts(dropna=False).head(10))
        print()
        print("Âge :")
        age = pd.to_numeric(per["AGE"], errors="coerce")
        print("  moyenne=", round(age.mean(), 1), ", médiane=", round(age.median(), 1))
else:
    print("Aucune donnée FARS chargée.")

## 3. Focus Tesla dans FARS 2024

Si des véhicules Tesla sont présents, exploration de leur gravité, vitesse, conditions.

In [ ]:
if fars and "vehicle" in fars and "person" in fars:
    veh = fars["vehicle"]
    per = fars["person"]
    is_tesla = veh["MAKENAME"].str.contains("Tesla", case=False, na=False)

    if is_tesla.sum() > 0:
        tesla_veh = veh[is_tesla]
        tesla_st_cases = set(tesla_veh["ST_CASE"].dropna().unique())

        tesla_per = per[per["ST_CASE"].isin(tesla_st_cases)]
        print("Personnes impliquées dans accidents Tesla :", len(tesla_per))
        print()
        print("Gravité (personnes, accidents Tesla) :")
        print(tesla_per["INJ_SEVNAME"].value_counts(dropna=False))

        sp_col = "SV Precrash Speed (MPH)"
        if sp_col in tesla_veh.columns:
            sp = pd.to_numeric(tesla_veh[sp_col], errors="coerce")
            print()
            print("Vitesse Tesla avant crash : moyenne=", round(sp.mean(), 1), "mph, médiane=", round(sp.median(), 1), "mph")

        if acc is not None:
            tesla_acc = acc[acc["ST_CASE"].isin(tesla_st_cases)]
            print()
            print("Accidents mortels impliquant Tesla :", len(tesla_acc))
            print("Décès dans ces accidents :", tesla_acc["FATALS"].astype(float).sum())
    else:
        print("Aucun véhicule Tesla trouvé dans FARS 2024.")
else:
    print("Données FARS incomplètes pour le focus Tesla.")

## 4. Visualisations comparatives (si données disponibles)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Répartition SGO par engagement ADAS
if sgo and any("Automation System Engaged?" in df.columns for df in sgo.values()):
    for name, df in sgo.items():
        if "Automation System Engaged?" in df.columns:
            vc = df["Automation System Engaged?"].value_counts(dropna=False)
            axes[0].bar(vc.index, vc.values, color=["#9aa3ad", "#be1e2d"][:len(vc)])
            axes[0].set_title("SGO " + name + " — Automatisation engagée")
            axes[0].tick_params(axis="x", rotation=45)
            break
else:
    axes[0].text(0.5, 0.5, "Données SGO non disponibles", ha="center", va="center")
    axes[0].set_title("SGO — Automatisation")

# 2. FARS véhicules par marque (top 10)
if fars and "vehicle" in fars:
    top_makes = fars["vehicle"]["MAKENAME"].value_counts(dropna=False).head(10)
    axes[1].bar(range(len(top_makes)), top_makes.values, color="#be1e2d")
    axes[1].set_xticks(range(len(top_makes)))
    axes[1].set_xticklabels([m[:15] for m in top_makes.index], rotation=45, ha="right")
    axes[1].set_title("FARS 2024 — Véhicules par marque (top 10)")
    axes[1].set_ylabel("Nombre")
else:
    axes[1].text(0.5, 0.5, "Données FARS non disponibles", ha="center", va="center")
    axes[1].set_title("FARS — Marques")

# 3. Gravité FARS (personnes)
if fars and "person" in fars:
    vc = fars["person"]["INJ_SEVNAME"].value_counts(dropna=False)
    axes[2].bar(vc.index, vc.values, color="#9aa3ad")
    axes[2].set_title("FARS 2024 — Gravité des blessures")
    axes[2].tick_params(axis="x", rotation=45)
else:
    axes[2].text(0.5, 0.5, "Données FARS non disponibles", ha="center", va="center")
    axes[2].set_title("FARS — Gravité")

plt.tight_layout()
plt.show()

## 5. Hypothèse de transférabilité USA → Europe

> **Hypothèse :** les données NHTSA (USA) décrivent un contexte routier, climatique et réglementaire différent de l'Europe.
> **Conséquence :** aucune conclusion tarifaire pour la Belgique/UE ne peut être tirée directement de cet EDA.
> Une récalibration sur des données européennes sera nécessaire avant toute utilisation en assurance auto européenne.

> ⚠️ Limitation : les données FARS ne contiennent que des accidents mortels (pas d'exposition kilométrique). Les données SGO/ADAS sont des rapports d'incidents sans kilométrage par véhicule. Un taux d'accident par 100 000 km ne peut donc pas être calculé directement depuis ces seules sources.

In [ ]:
# Résumé EDA pour documentation
eda_summary = {
    "sources_sgo": list(sgo.keys()) if sgo else [],
    "sources_fars": list(fars.keys()) if fars else [],
    "n_sgo_incidents": sum(len(df) for df in sgo.values()) if sgo else 0,
    "n_fars_accidents": len(fars.get("accident", [])) if fars else 0,
    "n_fars_vehicles": len(fars.get("vehicle", [])) if fars else 0,
    "n_fars_persons": len(fars.get("person", [])) if fars else 0,
    "annee_fars": 2024,
}
print(json.dumps(eda_summary, indent=2, ensure_ascii=False))